In [ ]:
# Import or install Sionna
import sionna
import sionna.rt
from datetime import datetime
import threading
import os
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline
no_preview = False # Toggle to False to use the preview widget

# Import relevant components from Sionna RT
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, Camera,\
                      PathSolver, RadioMapSolver, subcarrier_frequencies
import mitsuba as mi

In [23]:

tapls=[]
txp=[]
rxp=[]
f=[]
xml=[]
path='C:/Users/sbjoh/OneDrive/Desktop/NSF/roomsMO_new_spheres/' #path to working directory

In [ ]:
def runtest(start,l,bw,abs,frange):
    for i in range(l):
        i=i+start
        sname='{1}nosphere{0}.xml'.format(1,path)
        scene = load_scene(sname, merge_shapes=False)
        #scene.get("elm__59").velocity = [1, 0, 0]
        scene.frequency = np.random.uniform(frange[0], frange[1], 1)
        scene.tx_array = PlanarArray(num_rows=1,
                                    num_cols=1,
                                    vertical_spacing=0.5,
                                    horizontal_spacing=0.5,
                                    pattern="dipole",
                                    polarization="V")

        # Configure antenna array for all receivers
        scene.rx_array = PlanarArray(num_rows=1,
                                    num_cols=1,
                                    vertical_spacing=0.5,
                                    horizontal_spacing=0.5,
                                    pattern="dipole",
                                    polarization="V")

        # Create transmitter
        txloc=np.load("{0}/txloc.npy".format(path), allow_pickle=True)
        txps=list(txloc[i])
        tx = Transmitter(name="tx",
                        position=txps,
                        display_radius=2)

        # Add transmitter instance to scene
        scene.add(tx)
        # Create a receiver
        rxloc=np.load("{0}/rxloc.npy".format(path), allow_pickle=True)
        rxps=list(rxloc[i])
        rx = Receiver(name="rx",
                    position=rxps,
                    display_radius=2)
        # Add receiver instance to scene
        scene.add(rx)
        tx.look_at(rx) 
        # Instantiate a path solver
        # The same path solver can be used with multiple scenes
        p_solver  = PathSolver()
        # Compute propagation paths
        paths = p_solver(scene=scene,
                        max_depth=5,
                        los=True,
                        specular_reflection=True,
                        diffuse_reflection=False,
                        refraction=True,
                        synthetic_array=False,
                        seed=41)
        #below is channel impulse response as opposed to taps (unflitered)
        #a, tau = paths.cir(normalize_delays=True, out_type="numpy")
        #this returns  [num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths, num_time_steps],
        #t = tau[0,0,0,0,:]/(1/bw) # time scale
        #a_abs = np.abs(a)[0,0,0,0,:,0] #amplitudes complex
        taps = paths.taps(bandwidth=bw, # Bandwidth to which the channel is low-pass filtered
                        l_min=-6,        # Smallest time lag
                        l_max=100,       # Largest time lag
                        sampling_frequency=None, # Sampling at Nyquist rate, i.e., 1/bandwidth
                        normalize=True,  # Normalize energy
                        normalize_delays=True,
                        out_type="numpy")
        if(abs):
            taps=abs(taps)
        tapls.append((taps)[0,0,0,0,0])
        txp.append(txps)
        rxp.append(rxps)
        f.append(scene.frequency)
        xml.append(sname)


In [25]:
runtest(0,5000,100e6,False,[10e9,10e9])



In [26]:

mega=[txp,rxp,xml,float(f[0][0]),tapls]
np.save('{0}/dataset_nosphere.npy'.format(path), np.array(mega, dtype=object), allow_pickle=True)